In [ ]:
!pip -q install wfdb pywavelets neurokit2

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/Diplom2')
RAW = BASE / 'data' / 'raw'
META = BASE / 'data' / 'meta'
PRE_DIR = BASE / 'data' / 'preprocessed'
ARCH = BASE / 'data' / 'archives'
META.mkdir(parents=True, exist_ok=True)
PRE_DIR.mkdir(parents=True, exist_ok=True)
ARCH.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'code'))

KeyboardInterrupt: 

In [ ]:
WORK = Path('/content/work')
EXTRACT = WORK / 'code15_extracted'
PREP = WORK / 'prepared'
CODE15_BASE_PREP = PREP / 'code15_base'
CODE15_ZIP_PREP = PREP / 'code15_zip'
SAMI_PREP = PREP / 'samitrop'
PTB_PREP = PREP / 'ptbxl'

shutil.rmtree(EXTRACT, ignore_errors=True)
shutil.rmtree(PREP, ignore_errors=True)
EXTRACT.mkdir(parents=True, exist_ok=True)
CODE15_BASE_PREP.mkdir(parents=True, exist_ok=True)
CODE15_ZIP_PREP.mkdir(parents=True, exist_ok=True)
SAMI_PREP.mkdir(parents=True, exist_ok=True)
PTB_PREP.mkdir(parents=True, exist_ok=True)

In [ ]:
CODE15_ZIP_DIR = RAW / 'code15_zip'
zips = sorted(CODE15_ZIP_DIR.glob('exams_part*.zip'))
print('zips found:', len(zips))
for z in zips:
    print('extracting', z.name)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(EXTRACT)
print('extracted to', EXTRACT)

NameError: name 'RAW' is not defined

In [ ]:
BASE_HDF5 = sorted((RAW / 'code15' / 'part0').glob('*.hdf5')) + sorted((RAW / 'code15' / 'part1').glob('*.hdf5'))
ZIP_HDF5 = sorted(EXTRACT.rglob('*.hdf5'))
code15_csvs = list((RAW / 'code15').glob('*.csv'))
print('base hdf5:', len(BASE_HDF5))
print('zip hdf5:', len(ZIP_HDF5))
print('code15 csvs:', code15_csvs)

In [ ]:
def _load_code15_csvs():
    label_df = None
    meta_df = None
    label_file = None
    meta_file = None
    for c in (RAW / 'code15').glob('*.csv'):
        df = pd.read_csv(c)
        cols = {x.lower() for x in df.columns}
        if 'chagas' in cols or 'chagas_label' in cols or 'label' in cols:
            label_df = df
            label_file = c
        elif 'age' in cols and ('sex' in cols or 'is_male' in cols):
            meta_df = df
            meta_file = c
    return label_df, meta_df, label_file, meta_file

label_df, meta_df, label_file, meta_file = _load_code15_csvs()
print('label file:', label_file)
print('meta file:', meta_file)
print('label cols:', list(label_df.columns))
print('meta cols:', list(meta_df.columns))

In [ ]:
def _norm_id_col(df):
    for c in df.columns:
        if c.lower() in ('exam_id', 'id'):
            return c
    return df.columns[0]

lid = _norm_id_col(label_df)
mid = _norm_id_col(meta_df)
label_df = label_df.rename(columns={lid: 'exam_id'})
meta_df = meta_df.rename(columns={mid: 'exam_id'})

label_col_candidates = [c for c in label_df.columns if c.lower() in ('chagas', 'label', 'chagas_label')]
label_df = label_df.rename(columns={label_col_candidates[0]: 'chagas'})
label_df['chagas'] = label_df['chagas'].astype(int)

code15_meta = meta_df.merge(label_df[['exam_id', 'chagas']], on='exam_id', how='inner')
print(code15_meta.shape, code15_meta['chagas'].value_counts().to_dict())

In [ ]:
import h5py
rng = np.random.RandomState(42)
zip_ids = set()
for h5 in tqdm(ZIP_HDF5, desc='zip exam ids'):
    with h5py.File(h5, 'r') as f:
        zip_ids.update(int(x) for x in f['exam_id'][:])
zip_meta = code15_meta[code15_meta['exam_id'].isin(zip_ids)].reset_index(drop=True)
pos_ids = zip_meta.loc[zip_meta['chagas'] == 1, 'exam_id'].to_numpy()
neg_ids = zip_meta.loc[zip_meta['chagas'] == 0, 'exam_id'].to_numpy()
neg_keep = rng.permutation(neg_ids)[:25000]
zip_keep = np.concatenate([pos_ids, neg_keep])
labels_all_path = WORK / 'code15_labels_all.csv'
labels_zip_path = WORK / 'code15_labels_zip.csv'
label_df[['exam_id', 'chagas']].to_csv(labels_all_path, index=False)
label_df[label_df['exam_id'].isin(zip_keep)][['exam_id', 'chagas']].to_csv(labels_zip_path, index=False)
print('zip kept:', len(zip_keep), 'pos:', len(pos_ids), 'neg:', len(neg_keep))

In [ ]:
from prepare_code15_data import get_parser as get_code15_parser, run as run_prepare_code15

run_prepare_code15(get_code15_parser().parse_args([
    '-i', *[str(x) for x in BASE_HDF5],
    '-d', str(meta_file),
    '-l', str(labels_all_path),
    '-o', str(CODE15_BASE_PREP),
]))
run_prepare_code15(get_code15_parser().parse_args([
    '-i', *[str(x) for x in ZIP_HDF5],
    '-d', str(meta_file),
    '-l', str(labels_zip_path),
    '-o', str(CODE15_ZIP_PREP),
]))

In [ ]:
from prepare_samitrop_data import get_parser as get_samitrop_parser, run as run_prepare_samitrop

run_prepare_samitrop(get_samitrop_parser().parse_args([
    '-i', str(RAW / 'samitrop' / 'exams.hdf5'),
    '-d', str(RAW / 'samitrop' / 'exams.csv'),
    '-o', str(SAMI_PREP),
]))

In [ ]:
from prepare_ptbxl_data import get_parser as get_ptbxl_parser, run as run_prepare_ptbxl

run_prepare_ptbxl(get_ptbxl_parser().parse_args([
    '-i', str(RAW / 'ptb_xl' / 'records500'),
    '-d', str(RAW / 'ptb_xl' / 'ptbxl_database.csv'),
    '-o', str(PTB_PREP),
]))

In [ ]:
from metadata import build_metadata

code15_df = build_metadata({'code15': [str(CODE15_BASE_PREP), str(CODE15_ZIP_PREP)]}, tqdm_func=tqdm)
samitrop_meta = build_metadata({'samitrop': [str(SAMI_PREP)]}, tqdm_func=tqdm)
ptbxl_meta = build_metadata({'ptbxl': [str(PTB_PREP)]}, tqdm_func=tqdm)
print('code15', len(code15_df), 'samitrop', len(samitrop_meta), 'ptbxl', len(ptbxl_meta))

code15/code15_base:   0%|          | 0/39798 [00:00<?, ?it/s]

code15/code15_zip:   0%|          | 0/30299 [00:00<?, ?it/s]

samitrop/samitrop:   0%|          | 0/1631 [00:00<?, ?it/s]

ptbxl/ptbxl:   0%|          | 0/21843 [00:00<?, ?it/s]

code15 70097 samitrop 1631 ptbxl 21843


In [ ]:
common = ['record_name', 'record_path', 'dataset_group', 'age', 'sex_code',
          'chagas_label', 'fs_hz', 'duration_sec']
all_meta = pd.concat([
    code15_df[common],
    samitrop_meta[common],
    ptbxl_meta[common],
], ignore_index=True)
print(all_meta['dataset_group'].value_counts())
print(all_meta['chagas_label'].value_counts(dropna=False))

dataset_group
code15      70097
ptbxl       21843
samitrop     1631
Name: count, dtype: int64
chagas_label
0    85777
1     7794
Name: count, dtype: int64


In [ ]:
from preprocessing import filter_records

all_meta = all_meta.dropna(subset=['chagas_label']).reset_index(drop=True)
all_meta['chagas_label'] = all_meta['chagas_label'].astype(int)
filtered = filter_records(all_meta, age_min=0, age_max=120)

filter_records: input=93571 | target_duration=7s | age=[0, 120]
  code15               kept= 69374/70097  dropped_short=723   dropped_age=0
  samitrop             kept=  1545/1631   dropped_short=86    dropped_age=0
  ptbxl                kept= 21550/21843  dropped_short=0     dropped_age=293
  TOTAL                kept=92469/93571


In [ ]:
from preprocessing import preprocess_dataset, make_archive

SIG_RAW = PRE_DIR / 'signals_raw.npy'
META_RAW = META / 'meta_raw.csv'

preprocessed = preprocess_dataset(
    filtered,
    signals_path=SIG_RAW,
    meta_path=META_RAW,
)
print('signals saved at', SIG_RAW, 'shape on disk:', np.load(SIG_RAW, mmap_mode='r').shape)
print('meta saved at', META_RAW)
print('archive size:', make_archive(SIG_RAW, ARCH / 'signals_raw.tar'))

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print('cleared scratch:', WORK)